In [4]:
import pandas as pd
import numpy as np
import joblib
import dice_ml
from dice_ml import Dice
import warnings
import os
from tqdm import tqdm  # 진행상황 바 표시용 (없으면 pip install tqdm)

warnings.filterwarnings('ignore')

print("=" * 80)
print("Step 3: DiCE 반사실 설명 생성 (Genetic Method - 고성능 모드)")
print("=" * 80)

# ============================================================================
# 1. 데이터 및 모델 로드
# ============================================================================
print("\n[Step 1] 데이터 및 모델 로드")

# 1-1. 원본 데이터 로드
if os.path.exists('selected_data_for_modeling.csv'):
    df_original = pd.read_csv('selected_data_for_modeling.csv')
else:
    raise FileNotFoundError("'selected_data_for_modeling.csv' 파일이 없습니다.")

# 1-2. 모델 및 파라미터 로드
# Step 2에서 저장한 파일명(base_model_final.pkl) 사용
try:
    model = joblib.load('base_model_final.pkl')
    feature_names = joblib.load('selected_features_final.pkl')
    threshold = joblib.load('base_model_threshold_final.pkl')
    
    print(f"모델 로드 성공: base_model_final.pkl")
    print(f"임계값(Threshold): {threshold:.4f}")
    
except FileNotFoundError:
    raise FileNotFoundError("모델 파일(base_model_final.pkl)이 없습니다. Step 2를 먼저 실행하세요.")

# ============================================================================
# 2. 분석 대상 선정 (부도 기업 전수)
# ============================================================================
print("\n[Step 2] 분석 대상 선정")

# 전체 데이터 중 실제 부도(1)인 기업만 필터링
target_df = df_original[df_original['PERF_12M'] == 1].copy()
X_target = target_df[feature_names]
target_ids = target_df['ID'].values

print(f"분석 대상: 총 {len(X_target)}개 기업 (부도 전수)")

# ============================================================================
# 3. DiCE 객체 생성 (Genetic 사용)
# ============================================================================
print("\n[Step 3] DiCE 객체 초기화 (Genetic Algorithm)")

# Data 객체
d = dice_ml.Data(
    dataframe=df_original.drop('ID', axis=1), 
    continuous_features=feature_names, 
    outcome_name='PERF_12M'
)

# Model 객체
m = dice_ml.Model(model=model, backend='sklearn', model_type='classifier')

# Explainer 생성 - 핵심: method='genetic'
# Genetic 알고리즘은 최적의 해를 찾기 위해 진화 과정을 거치므로 
# Random보다 시간은 더 걸리지만, "No Counterfactuals found" 오류를 거의 내지 않습니다.
exp = Dice(d, m, method='genetic')

print("설정 완료: method='genetic' (성공률 최적화)")

# ============================================================================
# 4. 반사실 설명(CF) 생성 루프
# ============================================================================
print("\n[Step 4] 반사실 설명 생성 시작")
print("※ Genetic 방식은 정밀 계산을 하므로 시간이 다소 소요됩니다.")

cf_results_list = []
cf_failed_ids = []

# tqdm으로 진행률 표시
for i in tqdm(range(len(X_target)), desc="Processing"):
    try:
        # 현재 처리할 기업 데이터
        row = X_target.iloc[i]
        current_id = target_ids[i]
        
        # DiCE 입력 포맷
        query_instance = row.to_frame().T
        
        # 원본 부도 확률 계산
        original_prob = model.predict_proba(query_instance)[0, 1]
        
        # -----------------------------------------------------------
        # [수정] 변경 불가능한 변수(Immutable) 정의
        # -----------------------------------------------------------
        # 이름에 '_1'(전기)이 들어가거나, 과거 데이터를 의미하는 변수들
        immutable_features = [
            'FN1_11_1', 'FN1_13_1', 'FN2_1_1', 'FN2_3_2', 'FN2_5_1', 
            'FN2_10_1', 'FN3_11_1', 'FN1_24_1'
            # 위 변수명은 예시이며, 실제 feature_names 리스트에 있는 것 중 
            # '전기', '작년', '과거'에 해당하는 변수는 모두 포함해야 함
        ]
        
        # 전체 변수 중 immutable을 제외한 변수만 변경 허용
        features_to_vary = [f for f in feature_names if f not in immutable_features]

        # -----------------------------------------------------------
        # CF 생성 실행
        # -----------------------------------------------------------
        dice_exp = exp.generate_counterfactuals(
            query_instance,
            total_CFs=4,            
            desired_class=0,        
            features_to_vary=features_to_vary, # [수정] all -> 변경 가능 변수 리스트
            verbose=False,
            # Genetic 알고리즘이 변경 불가능한 변수는 건드리지 않게 됨
        )
        
        # 결과 추출
        cf_df = dice_exp.cf_examples_list[0].final_cfs_df
        
        if cf_df is not None and not cf_df.empty:
            # 생성된 4개의 CF 저장
            for cf_idx, cf_row in cf_df.iterrows():
                # CF의 예측 확률 계산
                cf_features = cf_row[feature_names].values.reshape(1, -1)
                cf_prob = model.predict_proba(cf_features)[0, 1]
                
                result_entry = {
                    'ID': current_id,
                    'CF_Number': cf_idx + 1,
                    'Original_Proba': original_prob,
                    'Target_Proba': cf_prob
                }
                
                # 변수별 값 저장 (Original, CF, Change)
                for feat in feature_names:
                    orig_val = row[feat]
                    cf_val = cf_row[feat]
                    result_entry[f'Original_{feat}'] = orig_val
                    result_entry[f'CF_{feat}'] = cf_val
                    result_entry[f'Change_{feat}'] = cf_val - orig_val
                
                cf_results_list.append(result_entry)
        else:
            cf_failed_ids.append(current_id)
            
    except Exception as e:
        # 에러 발생 시(못 찾음 포함) 기록하고 계속 진행
        cf_failed_ids.append(current_id)
        continue

# ============================================================================
# 5. 결과 저장
# ============================================================================
print("\n" + "=" * 80)
print("[Step 5] 결과 저장")
print("-" * 80)

final_cf_df = pd.DataFrame(cf_results_list)

# 통계 출력
total_target = len(X_target)
success_cnt = final_cf_df['ID'].nunique() if not final_cf_df.empty else 0
fail_cnt = len(cf_failed_ids)

print(f"총 분석 대상: {total_target}건")
print(f"성공 기업 수: {success_cnt}건 (성공률: {success_cnt/total_target*100:.1f}%)")
print(f"실패 기업 수: {fail_cnt}건")

if not final_cf_df.empty:
    output_filename = 'cf_results_full.csv'
    final_cf_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"\n성공 데이터 저장 완료: {output_filename}")
    
    # 샘플 출력
    print("\n[샘플 데이터 확인 (첫 번째 성공 기업)]")
    sample = final_cf_df.iloc[0]
    print(f"ID: {sample['ID']}")
    print(f"부도 확률 변화: {sample['Original_Proba']:.4f} -> {sample['Target_Proba']:.4f}")
else:
    print("\n[Warning] 생성된 CF가 없습니다. 모델이나 데이터를 확인해주세요.")

if cf_failed_ids:
    pd.DataFrame({'ID': cf_failed_ids}).to_csv('cf_failed_ids.csv', index=False)
    print("실패 목록 저장 완료: cf_failed_ids.csv")

print("\nStep 3 완료.")

Step 3: DiCE 반사실 설명 생성 (Genetic Method - 고성능 모드)

[Step 1] 데이터 및 모델 로드
모델 로드 성공: base_model_final.pkl
임계값(Threshold): 0.8609

[Step 2] 분석 대상 선정
분석 대상: 총 266개 기업 (부도 전수)

[Step 3] DiCE 객체 초기화 (Genetic Algorithm)
설정 완료: method='genetic' (성공률 최적화)

[Step 4] 반사실 설명 생성 시작
※ Genetic 방식은 정밀 계산을 하므로 시간이 다소 소요됩니다.


Processing: 100%|████████████████████████████████████████████████████████████████████| 266/266 [03:25<00:00,  1.29it/s]



[Step 5] 결과 저장
--------------------------------------------------------------------------------
총 분석 대상: 266건
성공 기업 수: 266건 (성공률: 100.0%)
실패 기업 수: 0건

성공 데이터 저장 완료: cf_results_full.csv

[샘플 데이터 확인 (첫 번째 성공 기업)]
ID: 51029.0
부도 확률 변화: 0.9944 -> 0.4478

Step 3 완료.
